In [ ]:
# Boilerplate

%load_ext autoreload
%autoreload 2
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import datetime

import numpy as np
np.set_printoptions(linewidth=5000)
import pandas as pd
pd.set_option("display.max_rows", 2000)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 2000)
from scipy.stats import norm

from pandas.tseries.offsets import CustomBusinessDay
from Otto.abacus.calendars.USEquityHolidayCalendar import USEquityHolidayCalendar

from bokeh.plotting import figure
from bokeh.io import show, output_file, output_notebook, push_notebook
from bokeh.layouts import gridplot, row, column
from bokeh.models import ColumnDataSource, HoverTool, BasicTicker, ColorBar, ColumnDataSource, LinearColorMapper, PrintfTickFormatter,LabelSet 
from bokeh.models.formatters import DatetimeTickFormatter
output_notebook()

import logging as Log
Log.basicConfig(level=Log.INFO)

In [ ]:
file_to_use = 'VX_futures_snapshot_2022.csv'

fut_quotes = pd.read_csv(file_to_use, parse_dates=['exp'])
fut_quotes['exp_plus_30'] = [x + datetime.timedelta(days=30) for x in fut_quotes['exp']]

bday = CustomBusinessDay(calendar=USEquityHolidayCalendar())

In [ ]:
# Calculate business days and a crude volatility time for each future

# your model may vary, but should be less than 1 
holiday_relative_weight = 0.3

def business_day_count(row):
    return len(pd.date_range(start=row['exp'], end=row['exp_plus_30'], freq=bday))
    
def vol_day_count(row):
    business_days = len(pd.date_range(start=row['exp'], end=row['exp_plus_30'], freq=bday))
    non_business_days = (row['exp_plus_30'] - row['exp']).days - business_days
    return 1.0 * business_days + holiday_relative_weight * non_business_days

    
fut_quotes['idx'] = np.arange(len(fut_quotes))
fut_quotes['nbday'] = fut_quotes.apply(vol_day_count, axis=1)   
business_days_per_month = np.max(fut_quotes.nbday)  # hope there's at least one month without holidays!

fut_quotes['scaled_fair'] = fut_quotes['fair'].values * np.sqrt(business_days_per_month / fut_quotes['nbday'].values)

display(fut_quotes)

In [ ]:
fig = figure(width=1200, height=800, title='VX term structure')
ds = ColumnDataSource(fut_quotes)

# fair
fig.line(x=fut_quotes.index, y=fut_quotes.fair, color='black', alpha=0.3, legend_label="fair")
fig.scatter(x=fut_quotes.index, y=fut_quotes.fair, color='black', alpha=0.3, legend_label="fair")
# bid
fig.line(x=fut_quotes.index, y=fut_quotes.ubpx, color='blue', alpha=0.3, legend_label="bid")
fig.scatter(x=fut_quotes.index, y=fut_quotes.ubpx, color='blue', alpha=0.3, legend_label="bid")
#ask
fig.line(x=fut_quotes.index, y=fut_quotes.uapx, color='orange', alpha=0.3, legend_label="ask")
fig.scatter(x=fut_quotes.index, y=fut_quotes.uapx, color='orange', alpha=0.3, legend_label="ask")

fig.line(x=fut_quotes.index, y=fut_quotes.scaled_fair, color='green', alpha=0.3, legend_label="adjusted")
fig.scatter(x=fut_quotes.index, y=fut_quotes.scaled_fair, color='green', alpha=0.3, legend_label="adjusted")

labels = LabelSet(x='idx', y='fair', text='sym', x_offset=10, y_offset=-20, source=ds)
fig.add_layout(labels)
fig.legend.click_policy="hide"
fig.legend.location = "bottom_right"

show(fig)